# 01 · Prepare the cereal analysis sample

**Question:** Which observations are suitable for comparing cereal prices and demand?

This notebook joins sales to product descriptions, calculates unit prices and revenue, and applies one final set of sample rules. The unit of observation is a store–product–week record.

**Inputs:** `data/raw/wcer.zip` (containing `wcer.csv`) and `data/raw/upccer.csv`.
**Outputs:** `data/processed/cereal_clean.parquet` and `cereal_analysis.parquet`.

Run cells from top to bottom. Raw data is not included in the repository; the README explains setup.

## 1. Load the source files

The ZIP is read directly without extracting a second copy of the sales file. Product descriptions use Latin-1 encoding. A compact input summary replaces repeated row previews; it is a loading check, not a data-quality verdict.

In [ ]:
from pathlib import Path

# Kernels may start in the repository root or the notebooks directory.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks").is_dir() or not (PROJECT / "README.md").is_file():
    raise RuntimeError("Start the notebook from the repository root or notebooks directory.")

import pandas as pd
import zipfile

sales_path = PROJECT / "data" / "raw" / "wcer.zip"
products_path = PROJECT / "data" / "raw" / "upccer.csv"

with zipfile.ZipFile(sales_path) as z:
    with z.open("wcer.csv") as f:
        sales = pd.read_csv(f)

products = pd.read_csv(products_path, encoding="latin1")

pd.DataFrame({
    "rows": [len(sales), len(products)],
    "columns": [sales.shape[1], products.shape[1]],
}, index=["sales", "products"])

## 2. Attach product information and standardize prices

`PRICE` is the price for `QTY` units, so `UNIT_PRICE = PRICE / QTY`. Revenue is `MOVE × UNIT_PRICE`, where `MOVE` is the number of units sold. The hexadecimal price and profit fields are redundant with their numeric counterparts.

The left join retains sales records even if metadata is missing. `validate="many_to_one"` stops the workflow if duplicate product keys would multiply sales rows. The preview shows the calculated fields together so their units can be checked.

In [ ]:
sales = sales.drop(columns=["PRICE_HEX", "PROFIT_HEX"])
sales = sales.merge(
    products[["UPC", "DESCRIP", "SIZE"]],
    on="UPC", how="left", validate="many_to_one",
)
sales["UNIT_PRICE"] = sales["PRICE"] / sales["QTY"]
sales["REVENUE"] = sales["MOVE"] * sales["UNIT_PRICE"]

print("Sales records without a product description:", sales["DESCRIP"].isna().sum())
sales[["DESCRIP", "MOVE", "QTY", "PRICE", "UNIT_PRICE", "REVENUE"]].head()

## 3. Apply the final sample rules once

Retain positive sales, positive unit prices, and observations with `OK == 1`, the dataset's quality flag. Four UPCs identified in the original metadata review represent non-cereal merchandise and are excluded explicitly below.

These rules preserve the project's original analysis sample. Zero-sales observations are excluded because the later log-demand model requires positive quantities; results therefore describe positive-sales records, not all demand occasions. Extreme prices or volumes are not trimmed from the primary sample solely because they are unusual.

`cereal_clean` removes only non-cereal merchandise. `analysis` also applies the price, sales, and quality rules. Keeping these distinct makes the two exported datasets' purposes explicit.

In [ ]:
non_cereal_upcs = [
    317,          # Tony the Tiger T-shirt
    3828125053,   # Dominick's T-size shirt
    3828125057,   # Dominick's T-size shirt
    4300099100    # Holiday Home magazine shipper
]

analysis = sales[
    (sales["MOVE"] > 0) &
    (sales["UNIT_PRICE"] > 0) &
    (sales["OK"] == 1) &
    (~sales["UPC"].isin(non_cereal_upcs))
].copy()


cereal_clean = sales[
    ~sales["UPC"].isin(non_cereal_upcs)
].copy()

pd.DataFrame({
    "records": [len(sales), len(cereal_clean), len(analysis)],
    "share_of_source": [1.0, len(cereal_clean) / len(sales), len(analysis) / len(sales)],
}, index=["source", "cereal merchandise", "analysis sample"])

## 4. Review quality and coverage before export

Missing promotion codes can be meaningful (no code recorded), so this check does not drop all rows containing missing values. Repeated store–UPC–week keys are reported rather than silently deduplicated: they may represent separate source records, and the SQL reporting step aggregates them.

Inspect missing product descriptions, unexpected sample coverage, and duplicate counts before using a new source dataset. The distribution summary provides a single check for unusual prices, sales, or accounting margins; it is not another filtering rule.

In [ ]:
print("Products:", analysis["UPC"].nunique())
print("Stores:", analysis["STORE"].nunique())
print("Weeks:", analysis["WEEK"].min(), "to", analysis["WEEK"].max())
print("Repeated store–UPC–week keys:", analysis.duplicated(["STORE", "UPC", "WEEK"]).sum())
print("Missing values by field:\n", analysis.isna().sum().to_string())

analysis[["MOVE", "QTY", "UNIT_PRICE", "REVENUE", "PROFIT"]].describe()

## 5. Save the two datasets

The processed directory is created if needed. Subsequent notebooks read `cereal_analysis.parquet`; `cereal_clean.parquet` retains the broader cereal-only sample for future checks. Run notebook 02 next.

In [ ]:
(PROJECT / "data" / "processed").mkdir(parents=True, exist_ok=True)

cereal_clean.to_parquet(
    PROJECT / "data" / "processed" / "cereal_clean.parquet",
    index=False
)

analysis.to_parquet(
    PROJECT / "data" / "processed" / "cereal_analysis.parquet",
    index=False
)

print("Files saved successfully.")